<a href="https://colab.research.google.com/github/MSK456/LLM-Engineering-Journey/blob/main/week3/assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U gradio transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [ ]:
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer

print("🔥 Downloading CPU-friendly Qwen Model (Takes a minute...)")

# 1. Switched to a smaller 1.5B parameter model so your Colab doesn't run out of RAM
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 2. Load Model strictly for CPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32, # CPUs prefer standard 32-bit math
    device_map="cpu"           # Force it to the CPU
)

# 3. The Dataset Generator Logic
def generate_dataset(topic, num_rows, file_format):
    print(f"🧠 Generating {num_rows} rows of {topic} data... (Warning: CPUs are slow!)")

    system_prompt = f"""You are a senior data engineer. Your job is to generate highly realistic, synthetic datasets.
You must generate exactly {num_rows} rows of data about "{topic}".
Output the data in strict {file_format} format.
Do NOT include any greetings, explanations, or markdown code blocks (like ```csv).
ONLY output the raw {file_format} text so it can be saved directly to a file."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Generate the {num_rows} rows of {topic} in {file_format} format now."}
    ]

    # Send inputs to CPU instead of CUDA
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cpu")

    # Generate the data
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800, # Lowered slightly so your CPU finishes faster
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode and clean up the output
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)]
    generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return generated_text.strip()


# 4. The Gradio UI
print("🎨 Launching Gradio UI...")
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Synthetic Dataset Generator (CPU Mode)")
    gr.Markdown("Generate custom datasets using Local AI (Qwen).")

    with gr.Row():
        with gr.Column():
            topic_input = gr.Textbox(label="Dataset Topic", placeholder="e.g., Book Inventory, Pizza Orders")
            rows_input = gr.Slider(minimum=1, maximum=10, value=5, step=1, label="Number of Rows")
            format_input = gr.Radio(["CSV", "JSON"], value="CSV", label="Output Format")
            generate_btn = gr.Button("Generate Dataset", variant="primary")

        with gr.Column():
            output_display = gr.Code(label="Generated Data", language="markdown")

    generate_btn.click(
        fn=generate_dataset,
        inputs=[topic_input, rows_input, format_input],
        outputs=output_display
    )

demo.launch(debug=True, share=True)

🔥 Downloading CPU-friendly Qwen Model (Takes a minute...)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🎨 Launching Gradio UI...


/tmp/ipykernel_824/290977145.py:56: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ed9eba9c2c5632b767.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🧠 Generating 7 rows of Book Inventory data... (Warning: CPUs are slow!)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ed9eba9c2c5632b767.gradio.live


In [ ]:
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("🔥 Downloading & Loading Phi-3 Model in 4-bit (This takes a minute...)")

# 1. Setup 4-bit Quantization (Saves Colab RAM!)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# 2. Load the Model & Tokenizer
model_id = "microsoft/Phi-3-mini-4k-instruct"

# Using the native Hugging Face implementation (no trust_remote_code needed)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config
)

# 3. The Dataset Generator Logic
def generate_dataset(topic, num_rows, file_format):
    print(f"🧠 Generating {num_rows} rows of {topic} data in {file_format} format...")

    # Prompt Engineering: Forcing the AI to act like a data engineer
    system_prompt = f"""You are a senior data engineer. Your job is to generate highly realistic, synthetic datasets.
You must generate exactly {num_rows} rows of data about "{topic}".
Output the data in strict {file_format} format.
Do NOT include any greetings, explanations, or markdown code blocks (like ```csv).
ONLY output the raw {file_format} text so it can be saved directly to a file."""

    messages = [
        {"role": "user", "content": system_prompt}
    ]

    # Convert chat template to a raw string first
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize the string into proper PyTorch tensors and move to GPU
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    # Generate the data
    with torch.no_grad():
        outputs = model.generate(
            **inputs,            # Unpack the dictionary containing input_ids and attention_mask
            max_new_tokens=1500, # Plenty of room to write the dataset
            temperature=0.7,     # High enough for variety, low enough to stay structured
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Calculate prompt length so we only decode the newly generated text
    input_length = inputs["input_ids"].shape[1]
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return generated_text.strip()


# 4. The Gradio UI
print("🎨 Launching Gradio UI...")
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Synthetic Dataset Generator")
    gr.Markdown("Generate custom datasets using Local AI (Phi-3). Perfect for testing, machine learning, and data science.")

    with gr.Row():
        with gr.Column():
            topic_input = gr.Textbox(label="Dataset Topic", placeholder="e.g., E-commerce Customer Reviews, Health Vitals, SaaS User Login Logs")
            rows_input = gr.Slider(minimum=1, maximum=20, value=5, step=1, label="Number of Rows")
            format_input = gr.Radio(["CSV", "JSON"], value="CSV", label="Output Format")
            generate_btn = gr.Button("Generate Dataset", variant="primary")

        with gr.Column():
            output_display = gr.Code(label="Generated Data", language="markdown")

    # Connect the button to the function
    generate_btn.click(
        fn=generate_dataset,
        inputs=[topic_input, rows_input, format_input],
        outputs=output_display
    )

# Launch with share=True so you get a public link!
demo.launch(debug=True, share=True)

🔥 Downloading & Loading Phi-3 Model in 4-bit (This takes a minute...)


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

🎨 Launching Gradio UI...


/tmp/ipykernel_824/3826596950.py:65: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f56de7c757e70f1574.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🧠 Generating 10 rows of Health Vitals data in JSON format...
